In [ ]:
import gstools as gs
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from context_flux_no.simulations.pde.shallow_water import ParametrizedShallowWater1D
from context_flux_no.simulations.utils import generate_dataset
from context_flux_no.waveforms.grf import GaussianRandomField
from context_flux_no.waveforms.step import (
    MultichannelWaveform,
    PeriodicRandomStepFunction1D,
)


jax.config.update("jax_default_device", jax.devices("gpu")[0])

In [ ]:
grf = GaussianRandomField(
    covariance_fns=[
        gs.Gaussian(dim=1, var=0.5, len_scale=0.3),
        gs.Gaussian(dim=1, var=0.5, len_scale=0.3),
    ],
    transforms=[gs.normalizer.LogNormal(), None],
)

x = jnp.arange(0.005, 1.005, 0.01)
u0 = grf.sample(x, jax.random.key(11))
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
for i, ax in enumerate(axes):
    ax.plot(x, u0[i])

In [ ]:
pde = ParametrizedShallowWater1D(0.5, 1.5, 12)
sol = pde.solve(
    lambda x: grf.sample(x, jax.random.key(0)),
    x_span=(0, 1),
    Nx=100,
    t_span=(0, 0.5),
    Nt=100,
    bc="periodic",
)

In [ ]:
sol

In [ ]:
sol["values"].shape

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 5), sharey="row")
for i in range(axes.shape[1]):
    axes[0, i].plot(x, sol["values"][0, 4 * i, 0])
    axes[1, i].plot(x, sol["values"][0, 4 * i, 1])

In [ ]:
dataset = generate_dataset(
    n_coeffs=10,
    n_ics_per_coeff=10,
    pde_factory=ParametrizedShallowWater1D,
    initial_condition_fn=GaussianRandomField(
        covariance_fns=[
            gs.Gaussian(dim=1, var=0.5, len_scale=0.3),
            gs.Gaussian(dim=1, var=0.5, len_scale=0.3),
        ],
        transforms=[gs.normalizer.LogNormal(), None],
    ).sample,
    coeff_range_dict={
        "gamma": (0.5, 1.5),
        "alpha": (0.5, 1.5),
        "beta": (8, 12),
    },
    x_span=(0, 1),
    Nx=100,
    t_span=(0, 0.5),
    Nt=100,
    seed=0,
)

In [ ]:
dataset

In [ ]:
step = MultichannelWaveform(
    [
        PeriodicRandomStepFunction1D(value_min=0.5, value_max=4.5),
        GaussianRandomField(
            covariance_fns=[
                gs.Gaussian(dim=1, var=0.5, len_scale=0.3),
            ],
        ),
    ]
)

x = jnp.arange(0.005, 1.005, 0.01)
u0 = step.sample(x, jax.random.key(0))
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
for i, ax in enumerate(axes):
    ax.plot(x, u0[i])

In [ ]:
pde = ParametrizedShallowWater1D(0.5, 1.5, 12)
sol = pde.solve(
    lambda x: step.sample(x, jax.random.key(0)),
    x_span=(0, 1),
    Nx=100,
    t_span=(0, 0.5),
    Nt=100,
    bc="periodic",
)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 5), sharey="row")
for i in range(axes.shape[1]):
    axes[0, i].plot(x, sol["values"][0, 4 * i, 0])
    axes[1, i].plot(x, sol["values"][0, 4 * i, 1])

In [ ]:
dataset = generate_dataset(
    n_coeffs=100,
    n_ics_per_coeff=100,
    pde_factory=ParametrizedShallowWater1D,
    initial_condition_fn=MultichannelWaveform(
        [
            PeriodicRandomStepFunction1D(value_min=0.5, value_max=4.5),
            GaussianRandomField(
                covariance_fns=[
                    gs.Gaussian(dim=1, var=0.5, len_scale=0.3),
                ],
            ),
        ]
    ).sample,
    coeff_range_dict={
        "gamma": (0.5, 1.5),
        "alpha": (0.5, 1.5),
        "beta": (8, 12),
    },
    x_span=(0, 1),
    Nx=100,
    t_span=(0, 0.5),
    Nt=100,
    seed=10,
)

In [ ]:
dataset

In [ ]:
dataset.to_netcdf(
    "../../data/shallow_water_1d_test_step_grf_seed=10.hdf5", engine="h5netcdf"
)

In [ ]:
dataset["values"].shape

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 5), sharey="row")
for i in range(axes.shape[1]):
    axes[0, i].plot(dataset.coords["x"], dataset["values"][0, 4 * i, 0])
    axes[1, i].plot(dataset.coords["x"], dataset["values"][0, 4 * i, 1])